<a href="https://colab.research.google.com/github/cerr/pycerr-notebooks/blob/main/01_getting_started/structure_operations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Working with structures: masks, boolean ops, margins & comparison

## Introduction

Structures (segmentations) are stored as contours and rasterized to binary masks on demand. This notebook shows how to get masks, combine them with **boolean operations**, grow a **margin** (e.g. GTV -> PTV), build a **label map**, and **compare** two structures with Dice and surface-distance metrics. We synthesize a couple of structures on the sample CT so it is self-contained.

## Install pyCERR

In [ ]:
!pip install "pyCERR @ git+https://github.com/cerr/pyCERR.git"

## Set up a demo plan

Load the sample CT and add two spherical structures, a `GTV` and a `Node`.

In [ ]:
import numpy as np
import cerr.plan_container as pc
from cerr import datasets
from cerr.contour import rasterseg as rs

planC = pc.loadDcmDir(datasets.fetch_sample_data('head_and_neck'))
nR, nC, nS = (int(v) for v in planC.scan[0].getScanSize())
xV, yV, zV = planC.scan[0].getScanXYZVals()
dx, dy, dz = abs(xV[1]-xV[0]), abs(yV[0]-yV[1]), abs(zV[1]-zV[0])
Xc, Yc, Zc = np.meshgrid(xV, yV, zV, indexing='xy')
cx, cy, cz = xV[nC//2], yV[nR//2], zV[nS//2]

def sphere(ox, oy, oz, r):
    return ((Xc-ox)**2 + (Yc-oy)**2 + (Zc-oz)**2) < r**2

planC = pc.importStructureMask(sphere(cx, cy, cz, 2.0).astype(np.uint8),
                               0, 'GTV', planC)
planC = pc.importStructureMask(sphere(cx+2.5, cy, cz, 1.5).astype(np.uint8),
                               0, 'Node', planC)
print([s.structureName for s in planC.structure])

## Structure masks

`getStrMask(structNum, planC)` returns the binary 3-D mask on the scan grid (the same shape as the scan array).

In [ ]:
gtvM = rs.getStrMask(0, planC).astype(bool)
nodM = rs.getStrMask(1, planC).astype(bool)
print('GTV voxels =', int(gtvM.sum()), '| Node voxels =', int(nodM.sum()))

## Boolean operations

Masks are NumPy arrays, so union/intersection/difference are just `|`, `&`, `& ~`. Bring a result back in as a structure with `importStructureMask`.

In [ ]:
union = gtvM | nodM
inter = gtvM & nodM
diff = gtvM & ~nodM
print('union=%d  intersection=%d  GTV-minus-Node=%d'
      % (union.sum(), inter.sum(), diff.sum()))
planC = pc.importStructureMask(union.astype(np.uint8), 0,
                               'GTV_U_Node', planC)

## Margin expansion (GTV -> PTV)

`surfaceExpand(mask, [dx, dy, dz], marginCm)` grows (or shrinks, with a negative margin) a mask by a physical margin in cm. Here we add a 5 mm isotropic margin to the GTV to make a PTV.

In [ ]:
from cerr.utils.mask import surfaceExpand
ptvM = surfaceExpand(gtvM, [dx, dy, dz], 0.5)
planC = pc.importStructureMask(ptvM.astype(np.uint8), 0,
                               'PTV (GTV+5mm)', planC)
print('PTV voxels =', int(ptvM.sum()), '(GTV was', int(gtvM.sum()), ')')

## Label map

`getLabelMap` writes selected structures into a single integer label volume (`dim=3`). Note: where structures overlap, the last-written label wins - use `dim=4` (a stack of binary masks) for overlapping/nested structures. Here we label the two disjoint-ish structures GTV and Node.

In [ ]:
from cerr.dataclasses import structure as structr
labelDict = {planC.structure[i].structureName: i+1 for i in (0, 1)}
labelMap, _ = structr.getLabelMap(planC, labelDict, [0, 1], dim=3)
print('label map', labelMap.shape, '-> labels', np.unique(labelMap).tolist())

## Compare two structures (auto vs reference)

A common task is scoring an auto-segmentation against a reference. pyCERR ships the `surface-distance` package; here we make a shifted copy of the GTV to play the role of an 'auto' contour and compute Dice, 95% Hausdorff distance and average surface distance (mm).

In [ ]:
import surface_distance as sd
autoM = np.zeros_like(gtvM)
autoM[3:, :, :] = gtvM[:-3, :, :]            # shift the GTV by 3 rows
spacing_mm = (dy*10, dx*10, dz*10)           # (rows=y, cols=x, slices=z)

surfDist = sd.compute_surface_distances(gtvM, autoM, spacing_mm)
dice = sd.compute_dice_coefficient(gtvM, autoM)
hd95 = sd.compute_robust_hausdorff(surfDist, 95)
asd = sd.compute_average_surface_distance(surfDist)
print('Dice = %.3f' % dice)
print('HD95 = %.2f mm' % hd95)
print('Avg surface dist = %.2f mm' % np.mean(asd))

## Notes

* Other mask utilities in `cerr.utils.mask`: `fillHoles`, `largestConnComps`, `morphologicalClosing`, `computeBoundingBox`.
* For auto-segmentation QA against reference contours, see also `04_registration/TG211_metrics.ipynb`.